# Advanced Grape Leaf Disease Analysis: End-to-End Integrated Pipeline

This notebook provides a research-ready automated pipeline for:
1. **Segmentation Module**:
   - *Traditional*: Unsupervised leaf isolation and lesion detection (with border highlighting).
   - *Deep Learning*: Architectures for U-Net, DeepLabV3+, and FCN-8s for future research extensions.
2. **Dataset Generation**: Processing raw images to create a segmented dataset in Google Drive.
3. **Ensemble Classification**: Training a Soft-Voting Ensemble (DenseNet121 + EfficientNet_B0) on segmented images.
4. **Comparative Evaluation**: Analyzing Baselines vs. Ensemble using Accuracy, Loss curves, Confusion Matrices, and ROC curves.
5. **Severity Analysis**: Calculating and saving infection percentages to a CSV file.
6. **Inference Pipeline**: Full visual diagnosis from raw image to final result with marked borders.

In [ ]:
# !pip install torch torchvision torchaudio scikit-learn matplotlib opencv-python pillow seaborn pandas

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import pandas as pd
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from PIL import Image
import copy
import shutil
from tqdm.auto import tqdm

# Optional: Mount Google Drive if using Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except:
    print("Not running in Google Colab or Drive mounting failed.")

## 1. Configuration and Persistence Logic

In [ ]:
# --- CONFIGURATION ---
RAW_DATA_PATH = '/content/drive/MyDrive/GrapeDataset/Raw' # Path to folder with 4 subfolders
SEGMENTED_DATA_PATH = '/content/drive/MyDrive/GrapeDataset/Segmented' # Output folder for isolated leaves
MODEL_SAVE_PATH = '/content/drive/MyDrive/GrapeDataset/Models' 
RESULTS_CSV_PATH = os.path.join(MODEL_SAVE_PATH, 'severity_results.csv')

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

def save_model(model, name):
    path = os.path.join(MODEL_SAVE_PATH, f"{name}.pth")
    torch.save(model.state_dict(), path)
    print(f"Model weights saved to {path}")

def load_model(model, name, device):
    path = os.path.join(MODEL_SAVE_PATH, f"{name}.pth")
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        model.to(device)
        print(f"Successfully loaded weights for {name}")
        return True
    return False

## 2. Segmentation Module

### 2.1 Traditional Unsupervised Segmentation
This section handles the automated isolation of leaves and detection of diseased spots without needing pre-trained masks.

In [ ]:
def isolate_leaf(image_path):
    """Removes background and returns RGB leaf image and binary mask."""
    image = cv2.imread(image_path)
    if image is None: return None, None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Segment green areas
    lower_green = np.array([20, 30, 30])
    upper_green = np.array([100, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    
    # Morphological cleaning
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    leaf_only = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
    return leaf_only, mask

def segment_disease_with_borders(leaf_rgb, mask):
    """Detects lesions and returns binary mask, contours, and severity %."""
    lab = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2Lab)
    _, a, _ = cv2.split(lab)
    a_blurred = cv2.GaussianBlur(a, (5, 5), 0)
    
    # Otsu thresholding for automated lesion detection
    _, disease_mask = cv2.threshold(a_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    disease_mask = cv2.bitwise_and(disease_mask, disease_mask, mask=mask)
    
    # Find Contours for border highlighting
    contours, _ = cv2.findContours(disease_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    leaf_area = np.sum(mask > 0)
    disease_area = np.sum(disease_mask > 0)
    severity = (disease_area / leaf_area) * 100 if leaf_area > 0 else 0
    
    return disease_mask, contours, severity

def run_automated_segmentation(src_root, dest_root):
    """Iterates through raw dataset and saves segmented leaf images."""
    if os.path.exists(dest_root): shutil.rmtree(dest_root)
    os.makedirs(dest_root, exist_ok=True)
    
    for cat in [d for d in os.listdir(src_root) if os.path.isdir(os.path.join(src_root, d))]:
        print(f"Segmenting Category: {cat}")
        os.makedirs(os.path.join(dest_root, cat), exist_ok=True)
        for img_name in tqdm(os.listdir(os.path.join(src_root, cat))):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                leaf, _ = isolate_leaf(os.path.join(src_root, cat, img_name))
                if leaf is not None:
                    Image.fromarray(leaf).save(os.path.join(dest_root, cat, img_name))

### 2.2 Deep Learning Segmentation Architectures
These models (U-Net, DeepLabV3+, FCN-8s) are implemented for your research framework.

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.enc1, self.enc2 = conv_block(in_channels, 64), conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        u1 = self.up1(e2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

def get_deeplabv3_plus(num_classes=1):
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

def get_fcn_8s(num_classes=1):
    model = models.segmentation.fcn_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

## 3. Classification Module (Baselines and Ensemble)

In [ ]:
def get_densenet_baseline(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

def get_efficientnet_baseline(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

class SoftEnsemble(nn.Module):
    def __init__(self, m1, m2):
        super().__init__()
        self.m1, self.m2 = m1, m2
    def forward(self, x):
        return (self.m1(x) + self.m2(x)) / 2

## 4. Training and Comparative Evaluation

In [ ]:
def train_model(model, loaders, criterion, optimizer, num_epochs=10, device='cuda'):
    model = model.to(device)
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    best_acc, best_wts = 0.0, copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            run_loss, run_corr = 0.0, 0
            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                run_loss += loss.item() * inputs.size(0)
                run_corr += torch.sum(preds == labels.data)
            
            epoch_loss = run_loss / len(loaders[phase].dataset)
            epoch_acc = run_corr.double() / len(loaders[phase].dataset)
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())
        print(f'Epoch {epoch}: Val Acc {best_acc:.4f}')
    
    model.load_state_dict(best_wts)
    return model, history

def evaluate_all(models_dict, loader, classes, device='cuda'):
    results = {}
    plt.figure(figsize=(10, 8))
    
    for name, model in models_dict.items():
        model.eval()
        y_true, y_pred, y_probs = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                outputs = model(inputs.to(device))
                y_true.extend(labels.numpy())
                y_pred.extend(torch.max(outputs, 1)[1].cpu().numpy())
                y_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
        
        results[name] = (np.array(y_true), np.array(y_pred))
        print(f"\n--- {name} Classification Report ---")
        print(classification_report(y_true, y_pred, target_names=classes))
        
        fpr, tpr, _ = roc_curve(np.eye(len(classes))[y_true].ravel(), np.array(y_probs).ravel())
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.2f})')
    
    plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.title('ROC Comparison'); plt.show()
    return results

## 5. Visual Pipeline Inference

In [ ]:
def visualize_diagnosis(image_path, model, classes, device='cuda'):
    orig = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    leaf, mask = isolate_leaf(image_path)
    d_mask, contours, severity = segment_disease_with_borders(leaf, mask)
    
    border_img = leaf.copy()
    cv2.drawContours(border_img, contours, -1, (255, 0, 0), 2)
    
    transform = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224, 224)), 
                                    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    with torch.no_grad():
        model.eval()
        out = model(transform(leaf).unsqueeze(0).to(device))
        pred_label = classes[torch.max(out, 1)[1][0]]
    
    fig, ax = plt.subplots(1, 4, figsize=(24, 6))
    ax[0].imshow(orig); ax[0].set_title("Original Image")
    ax[1].imshow(leaf); ax[1].set_title("1. Background Removed")
    ax[2].imshow(d_mask, cmap='hot'); ax[2].set_title("2. Disease Areas")
    ax[3].imshow(border_img)
    ax[3].set_title(f"3. Final: {pred_label}\nSeverity: {severity:.2f}% (Borders Marked)")
    plt.show()

## 6. End-to-End Execution

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Dataset Pre-processing
    if not os.path.exists(SEGMENTED_DATA_PATH) or not os.listdir(SEGMENTED_DATA_PATH):
        run_automated_segmentation(RAW_DATA_PATH, SEGMENTED_DATA_PATH)
    
    # 2. Load Data
    transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), 
                                    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    ds = datasets.ImageFolder(SEGMENTED_DATA_PATH, transform=transform)
    train_idx, temp_idx = train_test_split(np.arange(len(ds)), test_size=0.3, stratify=ds.targets, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[ds.targets[i] for i in temp_idx], random_state=42)
    
    loaders = {'train': DataLoader(Subset(ds, train_idx), batch_size=32, shuffle=True),
               'val': DataLoader(Subset(ds, val_idx), batch_size=32),
               'test': DataLoader(Subset(ds, test_idx), batch_size=32)}
    
    # 3. Model Setup
    d_base = get_densenet_baseline(len(ds.classes))
    e_base = get_efficientnet_baseline(len(ds.classes))
    
    # 4. Training / Persistence
    if not load_model(d_base, 'densenet_baseline', device):
        d_base, _ = train_model(d_base, loaders, nn.CrossEntropyLoss(), optim.AdamW(d_base.parameters(), lr=0.001))
        save_model(d_base, 'densenet_baseline')
        
    if not load_model(e_base, 'efficient_baseline', device):
        e_base, _ = train_model(e_base, loaders, nn.CrossEntropyLoss(), optim.AdamW(e_base.parameters(), lr=0.001))
        save_model(e_base, 'efficient_baseline')
    
    ensemble = SoftEnsemble(d_base, e_base)
    
    # 5. Comparative Evaluation
    evaluate_all({'Ensemble': ensemble, 'DenseNet121': d_base, 'EfficientNet_B0': e_base}, 
                 loaders['test'], ds.classes, device=device)
    
    # 6. Severity Logging to CSV
    print("Logging severity data...")
    log_data = []
    for path, lab_idx in ds.samples:
        leaf, mask = isolate_leaf(path)
        _, _, sev = segment_disease_with_borders(leaf, mask)
        log_data.append({'img': os.path.basename(path), 'class': ds.classes[lab_idx], 'sev': sev})
    pd.DataFrame(log_data).to_csv(RESULTS_CSV_PATH, index=False)
    
    # 7. Pipeline Visualization
    visualize_diagnosis(ds.samples[0][0], ensemble, ds.classes, device=device)

if __name__ == "__main__":
    main()